In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from DLAIUtils import Utils
import DLAIUtils

import os
import time
import torch

In [3]:
from tqdm.auto import tqdm

In [4]:
dataset = load_dataset("glue", "qqp", split="train[240000:290000]")

In [5]:
dataset[:5]

{'question1': ['Are cruise missiles fake?',
  'How do I start a designing business?',
  'I forgot my Facebook password and email password. How can I log into Facebook?',
  "What's the best monitor size for a 1920x1080 resolution?",
  "What does 'insight' mean? If I were to give my insights on a philosophical reading/topic, how should I structure my answer?"],
 'question2': ['Are missiles fake?',
  'How do I start an architecture design business?',
  'How do I recover my Facebook email and password?',
  'What is the best 1080p monitor?',
  'What does the KWL reading method mean and PQRST reading method mean? Please give examples on both each by reading and studying by a college textbook.'],
 'label': [0, 0, 1, 0, 0],
 'idx': [240000, 240001, 240002, 240003, 240004]}

In [6]:
questions = []

for q1, q2 in zip(dataset['question1'], dataset['question2']):
    questions.extend([q1, q2])

# Remove duplicates
questions = list(set(questions))

print('\n'.join(questions[:10]))
print('-' * 50)
print(f'Number of questions: {len(questions)}')

What are the characteristics of limestone?
Why is Switzerland Always Number One In HDI , Happiness , best To Born index?
Why isn't a war crime trial, Nüremberg style, being considered for George W. Bush?
Why does India like the English language? Why is it still the major connecting language in India?
I cant find a job suitable for me so I decided to become a YouTuber. Where or how can I start?
Who is that person in your life with whom you can share everything and why?
Is it alright to try to find a girlfriend on Quora?
What is best seminar topic for civil engineering?
Who would you like to meet in person before you die?
How do i prepare for UPSC exams?
--------------------------------------------------
Number of questions: 89025


In [11]:
#Check cuda and Setup the model

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    print('Sorry no cuda.')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

Sorry no cuda.


Loading weights: 100%|██████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7458.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
query = 'which city is the most populated in the world?'
xq = model.encode(query)
xq.shape

(384,)

In [9]:
#Setup Pinecone
utils = Utils()
PINECONE_API_KEY = utils.get_pinecone_api_key()

In [12]:
pinecone = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = utils.create_dlai_index_name('dl-ai')

if INDEX_NAME in [index.name for index in pinecone.list_indexes()]:
    pinecone.delete_index(INDEX_NAME)
print(INDEX_NAME)
pinecone.create_index(name=INDEX_NAME, 
    dimension=model.get_sentence_embedding_dimension(), 
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region='us-east-1'))

index = pinecone.Index(INDEX_NAME)
print(index)

dl-ai-gsxob-zuvw-gjpxy8atneq9oqk81nyna3pua


In [14]:
#Create Embeddings and Upsert to Pinecone
batch_size=200
vector_limit=10000

questions = questions[:vector_limit]

import json

for i in tqdm(range(0, len(questions), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(questions))
    # create IDs batch
    ids = [str(x) for x in range(i, i_end)]
    # create metadata batch
    metadatas = [{'text': text} for text in questions[i:i_end]]
    # create embeddings
    xc = model.encode(questions[i:i_end])
    # create records list for upsert
    records = zip(ids, xc, metadatas)
    # upsert to Pinecone
    index.upsert(vectors=records)

100%|███████████████████████████████████████████████████████████████████████████████████| 50/50 [01:03<00:00,  1.27s/it]


In [15]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Mar 2026 01:55:09 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '40'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10000}},
 'storageFullness': 0.0,
 'total_vector_count': 10000,
 'vector_type': 'dense'}

In [16]:
#Run Your Query
# small helper function so we can repeat queries later
def run_query(query):
  embedding = model.encode(query).tolist()
  results = index.query(top_k=10, vector=embedding, include_metadata=True, include_values=False)
  for result in results['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['text']}")

In [17]:
run_query('which city has the highest population in the world?')

0.64: Which is biggest city in india?
0.62: What is the most populated state in India?
0.59: Which are the top 10 largest cities of India by area?
0.57: What is the largest demographic in Quora?
0.54: Why is Perth one of the most liveable cities in the world?
0.54: What country in Europe has the most beautiful women?
0.53: Which is the most loved country in the world?
0.52: Which is the best city to live in united states?
0.49: What is Canada’s population?
0.48: Which cities span two continents?


In [18]:
query = 'how do i make chocolate cake?'
run_query(query)

0.64: How are chocolate raw materials obtained and how is chocolate made?
0.6: How do you make ice cream?
0.55: What are some teenage cake ideas?
0.51: How do we bake cake in microwave oven? What should be the perfect temperature?
0.5: What is the price range for Carlo's Bake Shop cake?
0.46: How do I process cassava into cassava flour?
0.45: What is the recipe for vattalappam?
0.44: Why does Cadbury Chocolate not available in US?
0.42: What's the recipe for the most delicious thing you've ever cooked?
0.41: Baking: Why is my cake undercooked in the middle while it is cooked on the sides?
